# Optimización de Hiperparámetros de Document Splitting en HayStack
## Experimento: Grid Search sobre `split_length` y `split_overlap`
 
**Corpus:** Informes de Gestión del Poder Ejecutivo al Congreso Nacional  
**Fuente:** https://www.argentina.gob.ar/jefatura/asuntos-estrategicos/relaciones-parlamentarias-e-institucionales/informes-al-congreso

---

## Motivación

En un pipeline RAG, el **Document Splitting** es uno de los pasos más críticos y subestimados. La elección del tamaño del fragmento (`split_length`) y del solapamiento (`split_overlap`) afecta directamente:

- **La recuperación:** ¿El retriever puede encontrar el fragmento que contiene la respuesta?
- **La generación:** ¿El LLM tiene suficiente contexto para generar una respuesta fiel y completa?

Los **Informes de Gestión al Congreso** son un corpus ideal para este experimento porque:
- Los informes recientes (2015-2023) son homogéneos en estructura → buena línea base
- Los informes de los 90 tienen estructura "rara" → prueba de robustez
- El lenguaje es administrativo/técnico, con prosa densa y datos presupuestarios intercalados
- Las respuestas a preguntas suelen requerir contexto multi-párrafo

**Pregunta de diseño central:**
> ¿Cuál es la combinación óptima de `split_length` y `split_overlap` para maximizar la recuperación y la calidad de respuesta en un corpus de documentos administrativos en español?

## 1. Espacio de Búsqueda (Grid Search)

Se propone un diseño factorial completo sobre las siguientes variables:

| Variable | Valores | Justificación |
|---|---|---|
| `split_length` (palabras) | 256, 512, 1024 | Rango típico en literatura RAG; los informes tienen párrafos densos |
| `split_overlap` (% del length) | 10%, 15%, 20% | El overlap preserva continuidad en cortes de tablas y enumeraciones |

Esto genera **9 combinaciones** a evaluar:

| Config | split_length | split_overlap (palabras) |
|---|---|---|
| C1 | 256 | 26 (~10%) |
| C2 | 256 | 38 (~15%) |
| C3 | 256 | 51 (~20%) |
| C4 | 512 | 51 (~10%) |
| C5 | 512 | 77 (~15%) |
| C6 | 512 | 102 (~20%) |
| C7 | 1024 | 102 (~10%) |
| C8 | 1024 | 154 (~15%) |
| C9 | 1024 | 205 (~20%) |

## 2. Corpus y Golden Dataset

### 2.1 Selección de documentos

- **7 informes en total:**
  - 5 informes recientes (2018-2023) → estructura homogénea
  - 2 informes de los 90 (1995-1997) → estructura heterogénea/rara

### 2.2 Construcción del Golden Dataset (Set de Evaluación)

El golden dataset consiste en **40-50 pares (pregunta, respuesta_esperada, contexto_fuente)** construidos manualmente a partir de los 7 documentos seleccionados.

**Tipos de preguntas (balanceadas):**

| Tipo | Ejemplo | Desafío para el splitter |
|---|---|---|
| **Dato puntual** | ¿Cuánto fue la inversión en salud en 2020? | El dato puede quedar en el borde de un chunk |
| **Lista/Enumeración** | ¿Cuáles fueron los 5 ejes de gestión del Ministerio X? | La lista puede quedar fragmentada |
| **Contexto multi-párrafo** | ¿Cómo evolucionó la deuda externa entre 2018 y 2022? | Requiere contexto amplio |
| **Tabla/Cifra presupuestaria** | ¿Cuál fue el presupuesto ejecutado en infraestructura? | Las tablas son muy sensibles al tamaño del chunk |

**Para cada par, se anota:**
- `question`: La pregunta
- `expected_answer`: La respuesta correcta (texto del documento)
- `source_doc`: Nombre del informe
- `source_page`: Número de página
- `source_paragraph`: Párrafo exacto (para calcular si el retriever lo encontró)

## 3. Pipeline HayStack

El pipeline tiene dos fases: **Indexación** y **Retrieval + Generación**.

In [ ]:
# Instalación de dependencias
# pip install haystack-ai sentence-transformers ragas

In [ ]:
# NOTA: Este notebook describe el experimento sin ejecutarlo.
# Las celdas de código muestran CÓMO se implementaría.

from haystack import Pipeline
from haystack.components.converters import PyPDFToDocument
from haystack.components.preprocessors import DocumentSplitter
from haystack.components.writers import DocumentWriter
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.document_stores.in_memory import InMemoryDocumentStore

def build_indexing_pipeline(split_length: int, split_overlap: int) -> Pipeline:
    """
    Construye el pipeline de indexación con los hiperparámetros dados.
    
    Args:
        split_length: Tamaño del fragmento en palabras.
        split_overlap: Solapamiento en palabras entre fragmentos consecutivos.
    """
    document_store = InMemoryDocumentStore()

    pipeline = Pipeline()
    pipeline.add_component("converter", PyPDFToDocument())
    pipeline.add_component(
        "splitter",
        DocumentSplitter(
            split_by="word",
            split_length=split_length,
            split_overlap=split_overlap
        )
    )
    pipeline.add_component(
        "embedder",
        SentenceTransformersDocumentEmbedder(
            model="intfloat/multilingual-e5-base"  # Modelo multilingüe, soporta español
        )
    )
    pipeline.add_component("writer", DocumentWriter(document_store=document_store))

    pipeline.connect("converter", "splitter")
    pipeline.connect("splitter", "embedder")
    pipeline.connect("embedder", "writer")

    return pipeline, document_store

In [ ]:
from pathlib import Path
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

def build_retrieval_pipeline(document_store, top_k: int = 5) -> Pipeline:
    """
    Construye el pipeline de retrieval para una document store ya indexada.
    """
    pipeline = Pipeline()
    pipeline.add_component(
        "query_embedder",
        SentenceTransformersTextEmbedder(model="intfloat/multilingual-e5-base")
    )
    pipeline.add_component(
        "retriever",
        InMemoryEmbeddingRetriever(document_store=document_store, top_k=top_k)
    )
    pipeline.connect("query_embedder.embedding", "retriever.query_embedding")
    return pipeline

## 4. Métricas de Evaluación

Se evalúan **dos dimensiones** independientes:

### 4.1 Eficacia del Retrieval (la más importante para el splitting)

Estas métricas miden si el retriever **encuentra** el fragmento correcto, sin involucrar al LLM.

**Hit Rate @ K (Recall@K)**
> ¿El fragmento que contiene la respuesta está entre los primeros K resultados?

$$\text{Hit Rate@K} = \frac{\text{preguntas donde el chunk correcto está en top-K}}{\text{total de preguntas}}$$

**MRR (Mean Reciprocal Rank)**
> Premia que el chunk correcto aparezca lo más arriba posible en el ranking.

$$\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$$

### 4.2 Calidad de la Generación (con RAGAS)

Estas métricas miden la calidad de la respuesta final del LLM dado el contexto recuperado.

| Métrica | Descripción | Qué revela sobre el splitting |
|---|---|---|
| **Faithfulness** | ¿La respuesta se deriva estrictamente del contexto? | Chunks pequeños → mayor riesgo de alucinación |
| **Answer Relevance** | ¿La respuesta es completa y pertinente? | Chunks grandes con ruido → respuestas difusas |
| **Context Precision** | ¿Los chunks recuperados son relevantes? | Chunks grandes → más ruido, menor precisión |

In [ ]:
def compute_hit_rate(golden_dataset: list[dict], retrieval_pipeline: Pipeline, k: int = 5) -> float:
    """
    Calcula el Hit Rate@K para un conjunto de preguntas.
    
    Args:
        golden_dataset: Lista de dicts con 'question' y 'source_paragraph'.
        retrieval_pipeline: Pipeline de retrieval ya construido.
        k: Número de documentos a recuperar.
    
    Returns:
        Hit Rate entre 0 y 1.
    """
    hits = 0
    for item in golden_dataset:
        result = retrieval_pipeline.run({"query_embedder": {"text": item["question"]}})
        retrieved_chunks = result["retriever"]["documents"]
        
        # Verificar si alguno de los chunks recuperados contiene el párrafo fuente
        found = any(
            item["source_paragraph"] in chunk.content
            for chunk in retrieved_chunks
        )
        if found:
            hits += 1
    
    return hits / len(golden_dataset)


def compute_mrr(golden_dataset: list[dict], retrieval_pipeline: Pipeline, k: int = 10) -> float:
    """
    Calcula el Mean Reciprocal Rank para un conjunto de preguntas.
    """
    reciprocal_ranks = []
    for item in golden_dataset:
        result = retrieval_pipeline.run({"query_embedder": {"text": item["question"]}})
        retrieved_chunks = result["retriever"]["documents"]
        
        rr = 0.0
        for rank, chunk in enumerate(retrieved_chunks, start=1):
            if item["source_paragraph"] in chunk.content:
                rr = 1.0 / rank
                break
        reciprocal_ranks.append(rr)
    
    return sum(reciprocal_ranks) / len(reciprocal_ranks)

## 5. Procedimiento Experimental Completo

In [ ]:
import itertools
import pandas as pd

# Espacio de búsqueda
SPLIT_LENGTHS = [256, 512, 1024]
OVERLAP_RATIOS = [0.10, 0.15, 0.20]

# Paths de los PDFs del corpus
PDF_PATHS = list(Path("inputs/").glob("*.pdf"))  # Ajustar según estructura del proyecto

# Golden dataset (cargado desde archivo anotado manualmente)
# golden_dataset = load_golden_dataset("golden_dataset.json")

results = []

for split_length, overlap_ratio in itertools.product(SPLIT_LENGTHS, OVERLAP_RATIOS):
    split_overlap = int(split_length * overlap_ratio)
    config_name = f"len={split_length}_ovlp={int(overlap_ratio*100)}%"
    print(f"\n--- Evaluando: {config_name} ---")

    # 1. Construir e indexar
    indexing_pipeline, doc_store = build_indexing_pipeline(split_length, split_overlap)
    # indexing_pipeline.run({"converter": {"sources": PDF_PATHS}})

    # 2. Construir retrieval
    retrieval_pipeline = build_retrieval_pipeline(doc_store, top_k=5)

    # 3. Medir métricas de retrieval
    # hit_rate = compute_hit_rate(golden_dataset, retrieval_pipeline, k=5)
    # mrr = compute_mrr(golden_dataset, retrieval_pipeline, k=10)

    # Valores simulados para ilustrar la estructura del experimento
    hit_rate = None
    mrr = None

    results.append({
        "config": config_name,
        "split_length": split_length,
        "split_overlap_pct": int(overlap_ratio * 100),
        "split_overlap_words": split_overlap,
        "hit_rate@5": hit_rate,
        "mrr": mrr,
    })

df_results = pd.DataFrame(results)
print("\n=== RESULTADOS ===")
print(df_results.to_string(index=False))

## 6. Visualización Esperada de Resultados

Una vez ejecutado el experimento, se esperan visualizaciones como las siguientes:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Datos simulados para ilustrar el tipo de resultado esperado
split_lengths = [256, 512, 1024]
overlaps = ["10%", "15%", "20%"]

# Hipótesis: 512 con 15% overlap debería rendir mejor en este corpus
hit_rate_matrix = np.array([
    [0.58, 0.62, 0.60],   # 256 palabras
    [0.71, 0.78, 0.74],   # 512 palabras  <- hipótesis: mejor resultado aquí
    [0.65, 0.68, 0.66],   # 1024 palabras
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Resultados Simulados del Grid Search (ILUSTRATIVO)", fontsize=14, fontweight='bold')

# Heatmap de Hit Rate@5
sns.heatmap(
    hit_rate_matrix,
    annot=True, fmt=".2f", cmap="YlGn",
    xticklabels=overlaps,
    yticklabels=[f"{l} palabras" for l in split_lengths],
    ax=axes[0], vmin=0.5, vmax=0.85
)
axes[0].set_title("Hit Rate @ 5")
axes[0].set_xlabel("split_overlap")
axes[0].set_ylabel("split_length")

# Bar chart comparativo
configs = [f"{l}w/{o}" for l in split_lengths for o in overlaps]
scores = hit_rate_matrix.flatten()
colors = ["#2ecc71" if s == scores.max() else "#3498db" for s in scores]

axes[1].bar(configs, scores, color=colors)
axes[1].set_title("Hit Rate @ 5 por Configuración")
axes[1].set_xlabel("Configuración (split_length / overlap)")
axes[1].set_ylabel("Hit Rate")
axes[1].set_ylim(0.4, 0.95)
axes[1].tick_params(axis='x', rotation=45)
axes[1].axhline(y=scores.max(), color='red', linestyle='--', alpha=0.5, label=f'Máximo: {scores.max():.2f}')
axes[1].legend()

plt.tight_layout()
plt.savefig("outputs/grid_search_results_simulados.png", dpi=150, bbox_inches='tight')
plt.show()
print("Nota: Estos datos son SIMULADOS para ilustrar el tipo de análisis esperado.")

## 7. Hipótesis y Justificación

**Hipótesis principal:**
> Un `split_length` de **512 palabras** con un `split_overlap` del **15%** (~77 palabras) superará a las demás configuraciones en términos de Hit Rate@5 y Faithfulness.

**Justificación:**

| Factor | Razonamiento |
|---|---|
| **512 > 256 palabras** | Los informes al Congreso tienen prosa administrativa densa; 256 palabras suelen cortar en medio de una explicación técnica o tabla presupuestaria |
| **512 < 1024 palabras** | Chunks de 1024 palabras mezclan múltiples temas (ej: logros del Ministerio A con cifras del Ministerio B), agregando ruido al retriever |
| **15% de overlap** | Previene la pérdida de información en los bordes sin duplicar excesivamente el contenido indexado (20% genera demasiada redundancia) |

**Hipótesis secundaria sobre tipo de pregunta:**
> Las preguntas de **dato puntual** se verán más beneficiadas por chunks pequeños (256), mientras que las de **contexto multi-párrafo** requerirán chunks de 512-1024. El análisis por tipo de pregunta revelará si una configuración única es suficiente o si convendría un enfoque híbrido.

---

## 8. Análisis de Trade-offs

```
Chunks pequeños (256 palabras)
  ✅ Mayor precisión para datos puntuales
  ✅ Menor ruido por chunk
  ❌ Pierde contexto necesario para respuestas complejas
  ❌ Mayor riesgo de alucinación (el LLM no tiene suficiente info)
  ❌ Más chunks → más costo de embedding y almacenamiento

Chunks grandes (1024 palabras)
  ✅ Preserva contexto amplio
  ✅ Menos chunks → menor costo computacional
  ❌ Un chunk puede contener múltiples temas → ruido para el retriever
  ❌ El LLM puede "distraerse" con información irrelevante dentro del chunk

Overlap alto (20%)
  ✅ Reduce la pérdida de información en bordes
  ❌ Duplica contenido → aumenta el tamaño del índice
  ❌ Puede confundir al retriever con chunks muy similares
```

---

## Resumen Ejecutivo

| Elemento | Detalle |
|---|---|
| **Herramienta** | HayStack `DocumentSplitter` (split_by="word") |
| **Tipo de experimento** | Grid Search factorial (3×3 = 9 configuraciones) |
| **Corpus** | 7 Informes de Gestión al Congreso (5 recientes + 2 de los 90) |
| **Golden Dataset** | 40-50 pares pregunta-respuesta anotados manualmente |
| **Métrica principal** | Hit Rate@5 (Recall del retriever) |
| **Métricas secundarias** | MRR, Faithfulness (RAGAS), Answer Relevance |
| **Hipótesis** | 512 palabras + 15% overlap es el punto de equilibrio óptimo para prosa administrativa |
| **Modelo de embedding** | `intfloat/multilingual-e5-base` (multilingüe, soporta español) |